# Module 2 Fire Event Analysis

What burned, how severely, when and whether Module 1's pre-fire predictors
saw any of it coming.

Three parts:

1. **Severity** from Sentinel-2 dNBR and RdNBR, computed here rather than taken
   from MTBS, so every processing choice is visible and challengeable
2. **Progression** from VIIRS active-fire detections, identifying the major run
   days
3. **Prediction test** pre-fire condition versus observed severity

Part 3 is the one worth reading carefully. For large wind-driven fires the
honest answer is often that fuels explained very little, because extreme wind and
drought overwhelm fuel structure. That is a finding, not a failed analysis, and
reporting it is what distinguishes this from a product that only shows the fires
where fuels mattered.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

import daear_toolkit as dt
from daear_toolkit import data_access, indicators, viz
from daear_toolkit import fire_access as fa
from daear_toolkit import fire_indicators as fi

REGION = dt.POUDRE_CAMERON_PEAK
BBOX = REGION.bbox

perimeters = fa.get_mtbs_perimeters(BBOX, year=2020, min_acres=1000)
print(f"{len(perimeters)} MTBS perimeters >= 1,000 acres in 2020")
if not perimeters.empty:
    name_col = next((c for c in perimeters.columns if "name" in c), None)
    print(perimeters[[c for c in (name_col, "acres", "burnbndac") if c in perimeters.columns]].head())

ValueError: Assigning CRS to a GeoDataFrame without a geometry column is not supported. Supply geometry using the 'geometry=' keyword argument, or by providing a DataFrame with column name 'geometry'

## Scene selection 

dNBR quality is governed by date selection far more than by algorithm. Two
things to get right:

**Match phenology.** The pre- and post-fire scenes should be at the same point
in the growing season. An "initial assessment" post-fire scene from October
compared against a July pre-fire scene mixes senescence into the burn signal and
inflates apparent low-severity area.

**Choose the assessment window deliberately.** *Initial assessment* (immediately
post-fire) captures maximum spectral change including char and ash that will wash
away. *Extended assessment* (one growing season later) captures delayed
mortality and is the standard for ecological effects. They give different maps of
the same fire.

Extended assessment is used here and matched anniversary windows, one year apart 
because the downstream questions in Modules 3 and 4 are about ecological effect
and watershed response, not immediate char.

In [ ]:
PRE_START, PRE_END = "2020-07-15", "2020-08-10"     # weeks before ignition
POST_START, POST_END = "2021-07-15", "2021-08-10"    # matched window, one year later

pre_scene = data_access.get_optical_scene(BBOX, start=PRE_START, end=PRE_END, max_cloud_pct=10)
post_scene = data_access.get_optical_scene(BBOX, start=POST_START, end=POST_END, max_cloud_pct=10)

pre_nbr = fi.nbr(pre_scene)
post_nbr = fi.nbr(post_scene)
dnbr_raw = fi.dnbr(pre_nbr, post_nbr)

print(f"Pre-fire  window: {PRE_START} to {PRE_END}")
print(f"Post-fire window: {POST_START} to {POST_END}  (extended assessment)")
print(f"Raw dNBR range: {float(dnbr_raw.min()):.0f} to {float(dnbr_raw.max()):.0f}")

## The phenological offset correction

Even matched-anniversary scenes differ: different snowpack, different spring
precipitation, different sun angle. That difference appears as a uniform
non-zero dNBR across unburned ground, and without correcting it, an entire fire
perimeter can appear to have burned at low severity when much of it did not.

The correction takes the median dNBR over unburned control
ground just outside the perimeter and subtract it. Skipping this step is the most
common reason a dNBR map overstates low-severity area.

In [ ]:
from rasterio.features import geometry_mask
from rasterio.transform import from_bounds

ydim, xdim = dnbr_raw.dims
ys, xs = dnbr_raw.coords[ydim].values, dnbr_raw.coords[xdim].values
transform = from_bounds(xs.min(), ys.min(), xs.max(), ys.max(), len(xs), len(ys))

inside = ~geometry_mask(perimeters.geometry, out_shape=dnbr_raw.shape, transform=transform, invert=False)
inside = xr.DataArray(inside, coords=dnbr_raw.coords, dims=dnbr_raw.dims)

# Control ground: outside the perimeter but nearby, so vegetation and aspect are
# comparable. Distant terrain would introduce its own systematic differences.
buffered = perimeters.copy()
buffered["geometry"] = perimeters.to_crs(epsg=5070).buffer(3000).to_crs("EPSG:4326")
near = ~geometry_mask(buffered.geometry, out_shape=dnbr_raw.shape, transform=transform, invert=False)
control = xr.DataArray(near, coords=dnbr_raw.coords, dims=dnbr_raw.dims) & ~inside

offset = fi.compute_dnbr_offset(dnbr_raw, control)
dnbr_corrected = dnbr_raw - offset
rdnbr = fi.rdnbr(pre_nbr, post_nbr, offset=offset)

print(f"Control area: {int(control.sum())} pixels in a 3 km ring outside the perimeter")
print(f"Phenological offset: {offset:+.1f} dNBR units")
print(f"  (a large offset means the two scenes are not well matched -- revisit the dates)")
print(f"Corrected dNBR inside perimeter: mean {float(dnbr_corrected.where(inside).mean()):.0f}")

In [ ]:
severity = fi.classify_severity(dnbr_corrected.where(inside), metric="dnbr")
severity_rd = fi.classify_severity(rdnbr.where(inside), metric="rdnbr")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
viz.plot_raster(dnbr_corrected.where(inside), title="dNBR (offset-corrected)", ax=axes[0], cmap="YlOrRd", vmin=-100, vmax=1000)
viz.plot_raster(severity, title="Severity class (dNBR, Key & Benson)", ax=axes[1], cmap=viz.SEVERITY_CMAP)
viz.plot_raster(severity_rd, title="Severity class (RdNBR, Miller & Thode)", ax=axes[2], cmap=viz.SEVERITY_CMAP)
for ax in axes:
    perimeters.boundary.plot(ax=ax, color="black", lw=0.7)
plt.tight_layout()
plt.savefig("../outputs/02_severity_maps.png", dpi=150)
plt.show()

summary_dnbr = fi.severity_summary(severity, pixel_area_ha=0.04)   # 20 m Sentinel-2
summary_rdnbr = fi.severity_summary(severity_rd, pixel_area_ha=0.04)
comparison = summary_dnbr[["label", "acres", "pct"]].merge(
    summary_rdnbr[["label", "acres", "pct"]], on="label", suffixes=("_dnbr", "_rdnbr"))
print(comparison)
print("\nThe two metrics disagree, and that disagreement is the honest uncertainty in")
print("any severity number. RdNBR normalizes by pre-fire biomass, so it reports more")
print("high severity in the sparser vegetation where dNBR under-reads. Quote one, show")
print("both, and say which thresholds were used.")
comparison.to_csv("../outputs/02_severity_comparison.csv", index=False)

# Persist for Modules 3 and 4. dNBR/Key & Benson is the version carried forward
# because it is the more widely reported convention; the RdNBR version is saved
# alongside so downstream work can test whether conclusions depend on the choice.
severity.to_netcdf("../outputs/02_severity_class.nc")
severity_rd.to_netcdf("../outputs/02_severity_class_rdnbr.nc")
dnbr_corrected.to_netcdf("../outputs/02_dnbr_corrected.nc")
print("\nSaved: 02_severity_class.nc (dNBR), 02_severity_class_rdnbr.nc, 02_dnbr_corrected.nc")

## Progression from VIIRS detections

NASA FIRMS 375 m VIIRS detections between ignition and containment, reconstructing
when the fire made its major runs.

Cameron Peak burned for over three months with several distinct wind-driven runs,
including a major eastward push in mid-October. Daily detection counts and
centroid displacement identify those events.

**Two things detections are not.** They are not a perimeter, they are thermal
anomalies at overpass times, roughly twice daily, so a fast night run between
overpasses leaves no trace. And they are not severity so detection density
reflects fire radiative power and smoke transmissivity, not effects on the
ground.

In [ ]:
detections = fa.active_fire_series(BBOX, start="2020-08-13", end="2020-12-05", sensor="VIIRS_SNPP_SP")
print(f"{len(detections)} VIIRS detections between ignition and containment")

progression = fi.progression_from_detections(detections, freq="D")
progression.to_csv("../outputs/02_fire_progression.csv")

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
axes[0].bar(progression.index, progression["detections"], color="#b5443a", width=1.0)
axes[0].set_ylabel("VIIRS detections/day")
ax2 = axes[0].twinx()
ax2.plot(progression.index, progression["cumulative_detections"], color="black", lw=1.5)
ax2.set_ylabel("cumulative")

axes[1].plot(progression.index, progression["centroid_shift_km"], "o-", ms=3, color="#3b6ea5")
axes[1].set_ylabel("daily centroid shift (km)")
axes[1].set_xlabel("date")

top = progression.nlargest(5, "detections")
for d in top.index:
    for ax in axes:
        ax.axvline(d, color="orange", ls=":", alpha=0.8)

plt.tight_layout()
plt.savefig("../outputs/02_fire_progression.png", dpi=150)
plt.show()

print("\nFive highest-detection days (the major runs):")
print(top[["detections", "mean_frp", "max_frp", "centroid_shift_km"]].round(1))

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 7))
days_since = (pd.to_datetime(detections["acq_date"]) - pd.Timestamp("2020-08-13")).dt.days
sc = ax.scatter(detections["longitude"], detections["latitude"], c=days_since,
                s=4, cmap="inferno", alpha=0.6)
perimeters.boundary.plot(ax=ax, color="black", lw=0.9)
plt.colorbar(sc, ax=ax, label="days since ignition")
ax.set_title("Cameron Peak progression, VIIRS detections coloured by date")
plt.tight_layout()
plt.savefig("../outputs/02_progression_map.png", dpi=150)
plt.show()

## Did the pre-fire predictors work?

The test Module 1 exists to enable. Severity is binned against each pre-fire
predictor and the monotonic relationship is measured with Spearman appropriate
because severity class is ordinal and the relationship is monotonic at best.

Set expectations before looking: for a fire of this size and wind-driven
behaviour, weak relationships are the likely and honest outcome.

In [ ]:
pci = xr.open_dataarray("../outputs/01_prefire_condition_index.nc")
terrain = data_access.get_terrain(BBOX)
lf = fa.get_landfire(BBOX, layers=("fbfm40", "cc", "cbh", "cbd"), prefire=True)

predictors = {
    "prefire_condition_index": pci,
    "fuel_hazard": fi.fuel_hazard(lf["fbfm40"]),
    "crown_fire_potential": fi.crown_fire_potential(lf["cc"], lf["cbh"], lf["cbd"]),
    "canopy_cover": lf["cc"],
    "slope_deg": terrain["slope_deg"],
    "elevation": terrain["elevation"],
}

results, tables = [], {}
for name, layer in predictors.items():
    tab = fi.severity_vs_predictor(severity.where(inside), layer.where(inside), n_bins=10)
    if tab.empty:
        continue
    tables[name] = tab
    results.append({"predictor": name,
                    "spearman_rho": tab.attrs.get("spearman_rho"),
                    "p_value": tab.attrs.get("spearman_p"),
                    "pct_high_lowest_bin": tab.iloc[0]["pct_high_severity"],
                    "pct_high_highest_bin": tab.iloc[-1]["pct_high_severity"]})

prediction = pd.DataFrame(results).sort_values("spearman_rho", key=abs, ascending=False)
prediction.to_csv("../outputs/02_predictor_performance.csv", index=False)
prediction

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7.5))
for ax, (name, tab) in zip(axes.ravel(), tables.items()):
    ax.plot(tab["predictor_lo"], tab["pct_high_severity"], "o-", color="#b5443a")
    ax.set_title(f"{name}\nrho={tab.attrs.get('spearman_rho')}, p={tab.attrs.get('spearman_p')}", fontsize=9)
    ax.set_xlabel(name); ax.set_ylabel("% high severity")
plt.tight_layout()
plt.savefig("../outputs/02_predictor_performance.png", dpi=150)
plt.show()

best = prediction.iloc[0]
print(f"Strongest predictor: {best['predictor']}  (rho = {best['spearman_rho']})")
print(f"  High-severity fraction rises from {best['pct_high_lowest_bin']}% in the lowest bin")
print(f"  to {best['pct_high_highest_bin']}% in the highest.")
print("\nInterpret with care. A significant Spearman rho across hundreds of thousands of")
print("pixels is near-guaranteed and says little; spatial autocorrelation means the")
print("effective sample size is orders of magnitude smaller than the pixel count. Read")
print("the effect SIZE (the spread between the lowest and highest bins), not the p-value.")

## Summary

Severity computed two ways with a phenological offset correction, progression
reconstructed from VIIRS, and pre-fire predictors tested against observed
severity.

Module 3 uses `severity` for stratified recovery tracking. Module 4 uses it for
community exposure.

**What this analysis supports and what it does not:**

- **Supports:** relative severity patterns within this fire; timing of major
  runs; the direction and rough magnitude of fuel-severity relationships.
- **Does not support:** absolute severity comparison against other fires
  (thresholds are ecosystem-specific and uncalibrated here); any causal claim
  about fuel treatment effectiveness; severity at sub-20 m scale.
- **Needs field data:** Composite Burn Index plots would let the Key & Benson
  thresholds be replaced with locally calibrated ones. Without them, class
  boundaries are borrowed, and the acreage-by-class table inherits that
  uncertainty.

**On the p-values in Step 4.** With ~500,000 pixels, nearly any relationship
reaches significance. Spatial autocorrelation means neighbouring pixels are not
independent observations, so the effective sample size is far smaller than the
pixel count. Report effect sizes. A reviewer who knows this will check whether
you do.